In [ ]:
# Setup: rebuild the qPCR analysis directly from Supplemental Tables S1 and S2.
#
# External input workbooks:
#   circExor/experimental_validation/Supplemental_Table_S1.xlsx  (primer/target identity audit)
#   circExor/experimental_validation/Supplemental_Table_S2.xlsx  (raw Ct values)
#
# The 12 circExor prediction scores and prespecified EV-high/EV-low labels are fixed analysis
# metadata embedded below because neither supplemental workbook contains these prediction fields.
# Each plotting cell saves PNG, PDF, and SVG files and displays the figure.
#
# Color logic:
#   warm colors -> Cellular / dry EV-low or cell-enriched tendency
#   cool colors -> EVs / dry EV-high tendency

from pathlib import Path
import math
import statistics

import numpy as np
import openpyxl
from scipy import stats

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ---------- Paths ----------
NOTEBOOK_DIR = Path(".")
S1_XLSX = NOTEBOOK_DIR / "Supplemental_Table_S1.xlsx"
S2_XLSX = NOTEBOOK_DIR / "Supplemental_Table_S2.xlsx"
OUTDIR = NOTEBOOK_DIR / "qPCR_dry_wet_figures_warm_cool"
OUTDIR.mkdir(parents=True, exist_ok=True)

# ---------- User-provided core palettes ----------
warm_colors = [
    '#F8D7B4', '#F5D1AB', '#F2CBA2', '#F0C9A8', '#EDC39F', '#EABD96',
    '#E8BB99', '#E5B590', '#E2AF87', '#E0AD8A', '#DDA781', '#DAA178',
    '#D89F7B', '#D59972', '#D29369', '#D0916C', '#CD8B63', '#CA855A',
    '#C8835D', '#C57D54', '#C2774B', '#C0754E', '#BD6F45', '#BA693C',
    '#B8673F', '#B56136', '#B25B2D', '#B05930'
]

cool_colors = [
    '#D4E6F1', '#C5D9E8', '#B6CCDF', '#A7BFD6'
]

# Darker accents from the same palettes keep small points and lines legible.
COLORS = {
    "Dry EV-high": cool_colors[-1],
    "Dry EV-low": warm_colors[-6],
    "Cellular": warm_colors[-8],
    "EVs": cool_colors[-1],
}
LIGHT = {
    "Dry EV-high": cool_colors[1],
    "Dry EV-low": warm_colors[5],
    "Cellular": warm_colors[3],
    "EVs": cool_colors[0],
}

# ---------- Matplotlib style ----------
mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["axes.linewidth"] = 1.5
mpl.rcParams["xtick.major.width"] = 1.3
mpl.rcParams["ytick.major.width"] = 1.3
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# ---------- Small helpers ----------
def finite_float(value):
    try:
        value = float(value)
    except Exception:
        return None
    return value if math.isfinite(value) else None


def mean(values):
    values = [v for v in values if v is not None and math.isfinite(v)]
    return statistics.mean(values) if values else float("nan")


def sd(values):
    values = [v for v in values if v is not None and math.isfinite(v)]
    return statistics.stdev(values) if len(values) > 1 else float("nan")


def sem(values):
    s = sd(values)
    values = [v for v in values if v is not None and math.isfinite(v)]
    return s / math.sqrt(len(values)) if len(values) > 1 and math.isfinite(s) else float("nan")


def jitter(n, width=0.10):
    """Deterministic jitter so repeated runs make exactly the same scatter positions."""
    if n <= 1:
        return [0]
    return np.linspace(-width, width, n)


def save_and_show(fig, name):
    """Save one figure as both PNG and PDF, then display it in the notebook."""
    png = OUTDIR / f"{name}.png"
    pdf = OUTDIR / f"{name}.pdf"
    svg = OUTDIR / f"{name}.svg"
    fig.savefig(png, bbox_inches="tight", dpi=600)
    fig.savefig(pdf, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight", format="svg")
    print(f"Saved: {png}")
    print(f"Saved: {pdf}")
    print(f"Saved: {svg}")
    plt.show()


def rows_from_sheet(wb, sheet_name):
    ws = wb[sheet_name]
    headers = [c.value for c in ws[1]]
    rows = []
    for values in ws.iter_rows(min_row=2, values_only=True):
        if all(v is None for v in values):
            continue
        rows.append(dict(zip(headers, values)))
    return rows


def one_sided_p_from_stats(stat_rows, level, metric, test):
    for row in stat_rows:
        if row["analysis_level"] == level and row["metric"] == metric and row["test"] == test:
            return finite_float(row["p_one_sided"])
    return float("nan")


# ---------- Fixed dry-prediction metadata ----------
# These values were used to define the original dry/wet consistency analysis. They are embedded
# because Supplemental Tables S1/S2 contain qPCR data and primer information, but not circExor scores.
PREDICTION_METADATA = [
    {"target": "hsa_circ_0000788", "expected_class": "EV-high", "dry_score": 0.733526},
    {"target": "hsa_circ_0001336", "expected_class": "EV-high", "dry_score": 0.504043},
    {"target": "hsa_circ_0001703", "expected_class": "EV-high", "dry_score": 0.611477},
    {"target": "hsa_circ_0000253", "expected_class": "EV-high", "dry_score": 0.660675},
    {"target": "hsa_circ_0000036", "expected_class": "EV-high", "dry_score": 0.731131},
    {"target": "hsa_circ_0000658", "expected_class": "EV-high", "dry_score": 0.500315},
    {"target": "hsa_circ_0004623", "expected_class": "EV-low/cell-enriched", "dry_score": 0.329969},
    {"target": "hsa_circ_0084443", "expected_class": "EV-low/cell-enriched", "dry_score": 0.288280},
    {"target": "hsa_circ_0035654", "expected_class": "EV-low/cell-enriched", "dry_score": 0.393503},
    {"target": "hsa_circ_0001776", "expected_class": "EV-low/cell-enriched", "dry_score": 0.446256},
    {"target": "hsa_circ_0001222", "expected_class": "EV-low/cell-enriched", "dry_score": 0.367990},
    {"target": "hsa_circ_0005224", "expected_class": "EV-low/cell-enriched", "dry_score": 0.482003},
]
PREDICTION_BY_TARGET = {row["target"]: row for row in PREDICTION_METADATA}
TARGET_ORDER = [row["target"] for row in PREDICTION_METADATA]


def numeric_ct_values(row, prefix):
    values = []
    for tech_rep in (1, 2, 3):
        value = finite_float(row.get(f"{prefix} Ct {tech_rep}"))
        if value is not None and value != 0:
            values.append(value)
    return values


def parse_sample(sample):
    sample = str(sample).strip()
    group_text, rep_text = sample.rsplit(" ", 1)
    if group_text == "Cellular":
        sample_group = "Cellular"
    elif group_text == "EV":
        sample_group = "EVs"
    else:
        raise ValueError(f"Unexpected sample label in Supplemental Table S2: {sample!r}")
    return sample_group, int(rep_text)


# ---------- Read and validate Supplemental Tables ----------
s1_wb = openpyxl.load_workbook(S1_XLSX, data_only=True, read_only=True)
s1_rows = rows_from_sheet(s1_wb, "Primer_Information")
s1_targets = []
for row in s1_rows:
    target = str(row["Target Name"])
    if target.startswith("hsa_circ_") and target not in s1_targets:
        s1_targets.append(target)

s2_wb = openpyxl.load_workbook(S2_XLSX, data_only=True, read_only=True)
raw_ct_rows = rows_from_sheet(s2_wb, "Raw_Ct")
s2_targets = []
for row in raw_ct_rows:
    target = str(row["Target"])
    if target not in s2_targets:
        s2_targets.append(target)

assert s1_targets == TARGET_ORDER, (
    "S1 circRNA target order/content differs from embedded prediction metadata.\n"
    f"S1: {s1_targets}\nMetadata: {TARGET_ORDER}"
)
assert s2_targets == TARGET_ORDER, (
    "S2 target order/content differs from embedded prediction metadata.\n"
    f"S2: {s2_targets}\nMetadata: {TARGET_ORDER}"
)


# ---------- Rebuild sample-level ΔCt values from S2 Raw_Ct ----------
sample_rows = []
for row in raw_ct_rows:
    target = str(row["Target"])
    sample_group, bio_rep = parse_sample(row["Sample"])
    target_cts = numeric_ct_values(row, "Target")
    reference_cts = numeric_ct_values(row, "Reference")
    target_mean_ct = mean(target_cts) if target_cts else None
    reference_mean_ct = mean(reference_cts) if reference_cts else None
    delta_ct = (
        target_mean_ct - reference_mean_ct
        if target_mean_ct is not None and reference_mean_ct is not None
        else None
    )
    sample_rows.append({
        "target": target,
        "sample_group": sample_group,
        "bio_rep": bio_rep,
        "reference_target": str(row["Reference gene"]),
        "target_mean_ct_ignore0": target_mean_ct,
        "reference_mean_ct_ignore0": reference_mean_ct,
        "DeltaCt": delta_ct,
        "QC_flags": "PASS" if delta_ct is not None else "target_all_ND;DCt_ND",
    })


# ---------- Rebuild matched biological-replicate ΔΔCt values ----------
sample_lookup = {
    (row["target"], row["sample_group"], row["bio_rep"]): row
    for row in sample_rows
}

pairs = []
for target_order, target in enumerate(TARGET_ORDER, start=1):
    metadata = PREDICTION_BY_TARGET[target]
    group6 = "Dry EV-high" if metadata["expected_class"] == "EV-high" else "Dry EV-low"
    for bio_rep in (1, 2, 3):
        cellular = sample_lookup[(target, "Cellular", bio_rep)]
        evs = sample_lookup[(target, "EVs", bio_rep)]
        if cellular["DeltaCt"] is None or evs["DeltaCt"] is None:
            continue
        ddct = evs["DeltaCt"] - cellular["DeltaCt"]
        log2_ratio = -ddct
        pairs.append({
            "target_order": target_order,
            "target": target,
            "group6": group6,
            "bio_rep": bio_rep,
            "dry_score": metadata["dry_score"],
            "log2": log2_ratio,
            "ddCt": ddct,
            "RQ": 2 ** log2_ratio,
        })


# ---------- Rebuild Cellular-mean-calibrated RQ rows used by the supplemental RQ figure ----------
cal_rows_raw = []
for target in TARGET_ORDER:
    cellular_delta_ct = [
        row["DeltaCt"]
        for row in sample_rows
        if row["target"] == target
        and row["sample_group"] == "Cellular"
        and row["DeltaCt"] is not None
    ]
    cellular_mean_delta_ct = mean(cellular_delta_ct) if cellular_delta_ct else None
    for row in sample_rows:
        if row["target"] != target:
            continue
        log2_rq = (
            cellular_mean_delta_ct - row["DeltaCt"]
            if cellular_mean_delta_ct is not None and row["DeltaCt"] is not None
            else None
        )
        cal_rows_raw.append({
            **row,
            "Cellular_mean_DeltaCt_calibrator": cellular_mean_delta_ct,
            "DeltaDeltaCt_vs_CellMean": -log2_rq if log2_rq is not None else None,
            "log2_RQ_vs_CellMean": log2_rq,
            "RQ_vs_CellMean": 2 ** log2_rq if log2_rq is not None else None,
        })


# ---------- Target- and biological-replicate-level summaries ----------
targets = []
for target_order, target in enumerate(TARGET_ORDER, start=1):
    metadata = PREDICTION_BY_TARGET[target]
    valid_rows = [row for row in pairs if row["target"] == target]
    values = [row["log2"] for row in valid_rows]
    targets.append({
        "target_order": target_order,
        "target": target,
        "group6": "Dry EV-high" if metadata["expected_class"] == "EV-high" else "Dry EV-low",
        "dry_score": metadata["dry_score"],
        "mean_log2": mean(values) if values else None,
        "sd_log2": sd(values) if len(values) > 1 else None,
        "valid_pair_n": len(values),
    })

rep_rows = []
for rep in sorted({row["bio_rep"] for row in pairs}):
    high_values = [
        row["log2"] for row in pairs
        if row["bio_rep"] == rep and row["group6"] == "Dry EV-high"
    ]
    low_values = [
        row["log2"] for row in pairs
        if row["bio_rep"] == rep and row["group6"] == "Dry EV-low"
    ]
    rep_rows.append({
        "rep": rep,
        "high_mean": mean(high_values) if high_values else None,
        "low_mean": mean(low_values) if low_values else None,
    })

front_pair = [row["log2"] for row in pairs if row["group6"] == "Dry EV-high"]
back_pair = [row["log2"] for row in pairs if row["group6"] == "Dry EV-low"]
valid_targets = [row for row in targets if row["mean_log2"] is not None]
front_target = [row["mean_log2"] for row in valid_targets if row["group6"] == "Dry EV-high"]
back_target = [row["mean_log2"] for row in valid_targets if row["group6"] == "Dry EV-low"]
rep_high = [row["high_mean"] for row in rep_rows if row["high_mean"] is not None]
rep_low = [row["low_mean"] for row in rep_rows if row["low_mean"] is not None]

x_target = np.array([row["dry_score"] for row in valid_targets], dtype=float)
y_target = np.array([row["mean_log2"] for row in valid_targets], dtype=float)
x_pair = np.array([row["dry_score"] for row in pairs], dtype=float)
y_pair = np.array([row["log2"] for row in pairs], dtype=float)


# ---------- Recompute all statistics shown in the figure annotations ----------
welch_pair_p = stats.ttest_ind(
    front_pair, back_pair, equal_var=False, alternative="greater"
).pvalue
student_pair_p = stats.ttest_ind(
    front_pair, back_pair, equal_var=True, alternative="greater"
).pvalue
mw_pair_p = stats.mannwhitneyu(
    front_pair, back_pair, alternative="greater"
).pvalue
welch_target_p = stats.ttest_ind(
    front_target, back_target, equal_var=False, alternative="greater"
).pvalue
mw_target_p = stats.mannwhitneyu(
    front_target, back_target, alternative="greater"
).pvalue
paired_rep_p = stats.ttest_rel(
    rep_high, rep_low, alternative="greater"
).pvalue

pearson_target = stats.pearsonr(x_target, y_target, alternative="greater")
spearman_target = stats.spearmanr(x_target, y_target, alternative="greater")
pearson_pair = stats.pearsonr(x_pair, y_pair, alternative="greater")
spearman_pair = stats.spearmanr(x_pair, y_pair, alternative="greater")

print(f"S1 target audit passed: {len(s1_targets)} circRNA targets.")
print(f"Loaded {len(raw_ct_rows)} target × sample rows from S2 Raw_Ct.")
print(f"Rebuilt {len(pairs)} valid pair-level observations.")
print(f"Rebuilt {len(valid_targets)} valid target-level means.")
print(f"Target order: {', '.join(TARGET_ORDER)}")


In [ ]:
# Figure RQ: Recomputed RQ bar plots for all 12 RNA targets after excluding Ct=0.
#
# Bars = mean RQ, error bars = SD, dots = individual biological repeats.
# Cellular uses the warm palette; EVs uses the cool palette. The y-axis is log-scaled because RQ spans
# a large range, so this figure is supplemental rather than the main evidence figure.

from collections import defaultdict

rq_by = defaultdict(list)
notes_by_target = defaultdict(list)
for row in cal_rows_raw:
    target = str(row["target"])
    group = row["sample_group"]
    rq = finite_float(row["RQ_vs_CellMean"])
    if rq is not None:
        rq_by[(target, group)].append(rq)
    if row.get("QC_flags") and row.get("QC_flags") != "PASS":
        notes_by_target[target].append(row.get("QC_flags"))

fig, axes = plt.subplots(4, 3, figsize=(11, 14), dpi=300)
axes = axes.flatten()

for ax, target in zip(axes, TARGET_ORDER):
    groups = ["Cellular", "EVs"]
    for xi, group in enumerate(groups):
        values = rq_by.get((target, group), [])
        color = COLORS[group]
        light = LIGHT[group]
        if values:
            ax.bar(xi, mean(values), yerr=sd(values), color=light, edgecolor="black", linewidth=0.8, capsize=4, width=0.62)
            offsets = jitter(len(values), 0.06)
            ax.scatter([xi + o for o in offsets], values, s=28, color=color, edgecolor="black", linewidth=0.45, zorder=3)
        else:
            ax.text(xi, 1, "ND", ha="center", va="center", fontsize=8, fontweight="bold")
    ax.set_title(target, fontweight="bold")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(groups, rotation=20)
    ax.set_yscale("log")
    ax.set_ylabel("RQ")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

for ax in axes[len(TARGET_ORDER):]:
    ax.axis("off")

fig.suptitle("Supplemental RQ by target, Cellular calibrator, Ct=0 excluded", fontsize=14, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.97))
save_and_show(fig, "FigRQ_traditional_RQ_barplots_warm_cool")


In [ ]:
# Figure 1: Group scatter using all valid pair-level qPCR observations.
#
# Each point is one valid biological-repeat paired ΔΔCt result.
# y = log$_2$(EVs/Cellular) = -ΔΔCt.

fig, ax = plt.subplots(figsize=(5.2, 4.6))

for xi, (label, values) in enumerate([("Dry EV-high", front_pair), ("Dry EV-low", back_pair)]):
    offsets = jitter(len(values), 0.09)
    ax.scatter([xi + o for o in offsets], values, s=55, color=COLORS[label], edgecolor="black", linewidth=0.6, zorder=3, alpha=0.92)
    ax.hlines(mean(values), xi - 0.23, xi + 0.23, color="black", linewidth=2.0)
    ax.errorbar(xi, mean(values), yerr=sem(values), fmt="none", ecolor="black", elinewidth=1.5, capsize=5, capthick=1.5, zorder=4)

ax.set_xticks([0, 1])
ax.set_xticklabels([f"Dry EV-high\n(n={len(front_pair)})", f"Dry EV-low\n(n={len(back_pair)})"], fontweight="bold")
ax.set_ylabel("qPCR log$_2$(EVs/Cellular)")
ax.set_title("All valid paired qPCR enrichments", fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ymin = min(front_pair + back_pair)
ymax = max(front_pair + back_pair)
ax.set_ylim(ymin - 1.2, ymax + 1.8)
ax.text(
    0.5, ymax + 1.35,
    f"one-sided Welch p={welch_pair_p:.3g}\nStudent p={student_pair_p:.3g}; MW p={mw_pair_p:.3g}",
    ha="center", va="top", fontsize=9,
)

save_and_show(fig, "Fig1_all_valid_pairs_log2_group_scatter_warm_cool")


In [ ]:
import pandas as pd
import seaborn as sns

# Build the plotting DataFrame.
data = []
for val in front_pair:
    data.append({'qPCR log$_2$(EVs/Cellular)': val, 'Group': 'Dry EV-high'})
for val in back_pair:
    data.append({'qPCR log$_2$(EVs/Cellular)': val, 'Group': 'Dry EV-low'})
df_plot = pd.DataFrame(data)

def get_asterisks(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    return 'ns'

sns.set_theme(style="ticks", font_scale=1.1)
fig, ax = plt.subplots(figsize=(5.2, 4.6))

# Draw the box plot.
sns.boxplot(
    data=df_plot,
    x='Group',
    y='qPCR log$_2$(EVs/Cellular)',
    palette={'Dry EV-high': COLORS['Dry EV-high'], 'Dry EV-low': COLORS['Dry EV-low']},
    width=0.5,
    showfliers=False,  # Hide default outliers because points are overlaid separately.
    linewidth=1.5,
    ax=ax
)

# Preserve the original point style.
for xi, (label, values) in enumerate([("Dry EV-high", front_pair), ("Dry EV-low", back_pair)]):
    offsets = jitter(len(values), 0.09)
    ax.scatter([xi + o for o in offsets], values, s=55, color=COLORS[label], edgecolor="black", linewidth=0.6, zorder=3, alpha=0.92)

ax.set_xticks([0, 1])
ax.set_xticklabels([f"Predicted EV targets\n(n={len(front_pair)})", f"Predicted Cellular targets\n(n={len(back_pair)})"], fontweight="bold")
ax.set_ylabel("qPCR log$_2$(EVs/Cellular)")
ax.set_xlabel("")
# ax.set_title("All valid paired qPCR enrichments", fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Calculate significance brackets dynamically.
y_min = df_plot['qPCR log$_2$(EVs/Cellular)'].min()
y_max = df_plot['qPCR log$_2$(EVs/Cellular)'].max()
y_range = y_max - y_min
y_padding = y_range * 0.05
h = y_range * 0.015

x1, x2 = 0, 1
y_line = y_max + y_padding
ax.plot([x1, x1, x2, x2], [y_line, y_line+h, y_line+h, y_line], lw=1.2, color='black')
ax.text((x1+x2)/2, y_line+h, get_asterisks(welch_pair_p), ha='center', va='bottom', color='black', fontsize=12, fontweight='bold')

ymin_current, ymax_current = ax.get_ylim()
ax.set_ylim(ymin_current, max(ymax_current, y_line + y_padding * 4.5))
save_and_show(fig, "Fig1_all_valid_pairs_log2_group_scatter_warm_cool")


In [ ]:
# Figure 2: Group scatter using one qPCR mean per RNA target.
#
# This is more conservative than direct pooling because each RNA target contributes at most one point.
# Targets without valid qPCR target means are excluded from this plot.

fig, ax = plt.subplots(figsize=(5.2, 4.6))

groups = [
    ("Dry EV-high", [d for d in valid_targets if d["group6"] == "Dry EV-high"]),
    ("Dry EV-low", [d for d in valid_targets if d["group6"] == "Dry EV-low"]),
]

for xi, (label, rows) in enumerate(groups):
    values = [d["mean_log2"] for d in rows]
    offsets = jitter(len(values), 0.09)
    for off, d in zip(offsets, rows):
        ax.scatter(xi + off, d["mean_log2"], s=70, color=COLORS[label], edgecolor="black", linewidth=0.6, zorder=3)
        ax.text(xi + off, d["mean_log2"] + 0.35, d["target"], ha="center", fontsize=7)
    ax.hlines(mean(values), xi - 0.23, xi + 0.23, color="black", linewidth=2.0)
    ax.errorbar(xi, mean(values), yerr=sem(values), fmt="none", ecolor="black", elinewidth=1.5, capsize=5, capthick=1.5, zorder=4)

ax.set_xticks([0, 1])
ax.set_xticklabels([f"Dry EV-high\n(n={len(front_target)} targets)", f"Dry EV-low\n(n={len(back_target)} targets)"], fontweight="bold")
ax.set_ylabel("Mean qPCR log$_2$(EVs/Cellular) per target")
ax.set_title("Target-level qPCR enrichment", fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ymin = min(front_target + back_target)
ymax = max(front_target + back_target)
ax.set_ylim(ymin - 1.4, ymax + 2.4)
ax.text(0.5, ymax + 1.9, f"one-sided Welch p={welch_target_p:.3g}\nMann-Whitney p={mw_target_p:.3g}", ha="center", va="top", fontsize=9)

save_and_show(fig, "Fig2_target_mean_log2_group_scatter_warm_cool")


In [ ]:
import pandas as pd
import seaborn as sns

data_target = []
for val in front_target:
    data_target.append({'Mean qPCR log$_2$(EVs/Cellular)': val, 'Group': 'Dry EV-high'})
for val in back_target:
    data_target.append({'Mean qPCR log$_2$(EVs/Cellular)': val, 'Group': 'Dry EV-low'})
df_plot_target = pd.DataFrame(data_target)

def get_asterisks(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    return 'ns'

sns.set_theme(style="ticks", font_scale=1.1)
fig, ax = plt.subplots(figsize=(5.2, 4.6))

sns.boxplot(
    data=df_plot_target,
    x='Group',
    y='Mean qPCR log$_2$(EVs/Cellular)',
    palette={'Dry EV-high': COLORS['Dry EV-high'], 'Dry EV-low': COLORS['Dry EV-low']},
    width=0.5,
    showfliers=False,
    linewidth=1.5,
    ax=ax
)

groups = [
    ("Dry EV-high", [d for d in valid_targets if d["group6"] == "Dry EV-high"]),
    ("Dry EV-low", [d for d in valid_targets if d["group6"] == "Dry EV-low"]),
]

for xi, (label, rows) in enumerate(groups):
    values = [d["mean_log2"] for d in rows]
    offsets = jitter(len(values), 0.09)
    for off, d in zip(offsets, rows):
        ax.scatter(xi + off, d["mean_log2"], s=70, color=COLORS[label], edgecolor="black", linewidth=0.6, zorder=3, alpha=0.92)

ax.set_xticks([0, 1])
ax.set_xticklabels([f"Predicted EV targets\n(n={len(front_target)})", f"Predicted Cellular targets\n(n={len(back_target)})"], fontweight="bold")
ax.set_ylabel("Mean qPCR log$_2$(EVs/Cellular) per target")
ax.set_xlabel("")
# ax.set_title("Target-level qPCR enrichment", fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

y_min = df_plot_target['Mean qPCR log$_2$(EVs/Cellular)'].min()
y_max = df_plot_target['Mean qPCR log$_2$(EVs/Cellular)'].max()
y_range = y_max - y_min
y_padding = y_range * 0.05
h = y_range * 0.015

x1, x2 = 0, 1
y_line = y_max + y_padding
ax.plot([x1, x1, x2, x2], [y_line, y_line+h, y_line+h, y_line], lw=1.2, color='black')
ax.text((x1+x2)/2, y_line+h, get_asterisks(welch_target_p), ha='center', va='bottom', color='black', fontsize=12, fontweight='bold')

ymin_current, ymax_current = ax.get_ylim()
ax.set_ylim(ymin_current, max(ymax_current, y_line + y_padding * 4.5))

save_and_show(fig, "Fig2_target_mean_log2_group_scatter_warm_cool")

In [ ]:
# Figure 3: Biological-replicate mean plot.
#
# For each biological repeat, average log$_2$(EVs/Cellular) across valid EV-high targets and across
# valid EV-low/cell-enriched targets, then connect the two means.

fig, ax = plt.subplots(figsize=(5.2, 4.6))

for r in rep_rows:
    ax.plot([0, 1], [r["high_mean"], r["low_mean"]], color="#777777", linewidth=1.5, zorder=1)
    ax.scatter(0, r["high_mean"], s=70, color=COLORS["Dry EV-high"], edgecolor="black", linewidth=0.6, zorder=3)
    ax.scatter(1, r["low_mean"], s=70, color=COLORS["Dry EV-low"], edgecolor="black", linewidth=0.6, zorder=3)
    ax.text(-0.08, r["high_mean"], f"R{r['rep']}", ha="right", va="center", fontsize=8)

for xi, values in enumerate([rep_high, rep_low]):
    ax.hlines(mean(values), xi - 0.20, xi + 0.20, color="black", linewidth=2.2, zorder=4)
    ax.errorbar(xi, mean(values), yerr=sem(values), fmt="none", ecolor="black", elinewidth=1.5, capsize=5, capthick=1.5, zorder=4)

ax.set_xticks([0, 1])
ax.set_xticklabels(["Dry EV-high\nmean within rep", "Dry EV-low\nmean within rep"], fontweight="bold")
ax.set_ylabel("Replicate mean qPCR log$_2$(EVs/Cellular)")
ax.set_title("Biological-replicate mean enrichment", fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ymin = min(rep_high + rep_low)
ymax = max(rep_high + rep_low)
ax.set_ylim(ymin - 0.9, ymax + 1.5)
ax.text(0.5, ymax + 1.15, f"one-sided paired t-test p={paired_rep_p:.3g}", ha="center", va="top", fontsize=9)

save_and_show(fig, "Fig3_biological_replicate_mean_paired_warm_cool")


In [ ]:
# Figure 4: Dry score versus qPCR target-level EV enrichment.
#
# x = dry EV tendency score.
# y = target-level mean qPCR log$_2$(EVs/Cellular).

fig, ax = plt.subplots(figsize=(5.6, 4.8))
# Manual label offsets: "target": (x_offset, y_offset).
# Add targets here if labels overlap.
CUSTOM_OFFSETS = {
    "hsa_circ_0035654": (0, 0.55),
    "hsa_circ_0001336": (0.03, 0),
}

for d in valid_targets:
    ax.scatter(d["dry_score"], d["mean_log2"], s=80, color=COLORS[d["group6"]], edgecolor="black", linewidth=0.6, zorder=3)
    
    # Default offset.
    dx = 0.006
    dy = 0.2
    
    # Apply manual offsets when defined.
    if d["target"] in CUSTOM_OFFSETS:
        dx, dy = CUSTOM_OFFSETS[d["target"]]
        
    ax.text(d["dry_score"] + dx, d["mean_log2"] + dy, d["target"], fontsize=9)
coef = np.polyfit(x_target, y_target, 1)
xx = np.linspace(min(x_target) - 0.02, max(x_target) + 0.02, 100)
ax.plot(xx, coef[0] * xx + coef[1], color="black", linewidth=1.5, linestyle="--")

ax.set_xlabel("circExor EV-prediction score")
ax.set_ylabel("Mean qPCR log$_2$(EVs/Cellular) per target")
# ax.set_title("Dry prediction vs qPCR enrichment", fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

legend_labels = {
    "Dry EV-high": "Predicted EV targets",
    "Dry EV-low": "Predicted Cellular targets"
}

ax.legend(
    handles=[Patch(facecolor=COLORS[k], edgecolor="black", label=legend_labels[k]) for k in ["Dry EV-high", "Dry EV-low"]],
    frameon=False, loc="upper left",
)
ax.text(
    0.98, 0.04,
    f"Pearson r={pearson_target.statistic:.3f}, one-sided p={pearson_target.pvalue:.3g}\nSpearman rho={spearman_target.statistic:.3f}, one-sided p={spearman_target.pvalue:.3g}",
    transform=ax.transAxes, ha="right", va="bottom", fontsize=9,
)

save_and_show(fig, "Fig4_dry_score_vs_qPCR_target_mean_correlation_warm_cool")


In [ ]:
# Figure 5: Rank targets by dry EV tendency score and overlay qPCR enrichment.
#
# Bars = dry EV tendency score, colored by the prespecified EV-high/EV-low group.
# Black line/dots = qPCR target-level mean log$_2$(EVs/Cellular).
# ND marks targets lacking valid qPCR target-level values.

ranked = sorted(targets, key=lambda d: d["dry_score"], reverse=True)
labels = [d["target"] for d in ranked]
dry_values = [d["dry_score"] for d in ranked]
q_values = [d["mean_log2"] if d["mean_log2"] is not None else np.nan for d in ranked]
bar_colors = [COLORS[d["group6"]] for d in ranked]

fig, ax1 = plt.subplots(figsize=(8, 4))
x = np.arange(len(ranked))

ax1.bar(x, dry_values, color=bar_colors, edgecolor="black", linewidth=0.8, alpha=0.88)
ax1.set_ylabel("circExor EV-prediction score", fontweight="bold")
ax1.set_ylim(0, 0.85)
ax1.set_xticks(x)
ax1.set_xticklabels(labels, rotation=45, ha="right", fontweight="bold")

ax2 = ax1.twinx()
ax2.plot(x, q_values, color="black", marker="o", linewidth=1.8, label="qPCR mean log$_2$(EVs/Cellular)")
for xi, yi in zip(x, q_values):
    if not math.isfinite(yi):
        ax2.text(xi, 0.3, "", ha="center", va="bottom", fontsize=8, fontweight="bold")
ax2.set_ylabel("qPCR EV enrichment", fontweight="bold")

# ax1.set_title("Targets ranked by dry EV tendency", fontweight="bold")
ax1.spines["top"].set_visible(False)
ax2.spines["top"].set_visible(False)

from matplotlib.patches import Patch
custom_bars = [
    Patch(facecolor=COLORS["Dry EV-high"], edgecolor="black", alpha=0.88, label="Predicted EV targets (score)"),
    Patch(facecolor=COLORS["Dry EV-low"], edgecolor="black", alpha=0.88, label="Predicted Cellular targets (score)"),
]

lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(custom_bars + lines2, [h.get_label() for h in custom_bars] + labels2, frameon=False, loc="upper right")
fig.tight_layout()

save_and_show(fig, "Fig5_ranked_dry_score_and_qPCR_log2_warm_cool")


In [ ]:
# Figure 6: Exploratory pair-level dry score versus qPCR enrichment correlation.
#
# This uses every valid biological-repeat pair instead of target means. It gives more points but is
# exploratory because multiple observations come from the same target.

fig, ax = plt.subplots(figsize=(5.6, 4.8))

for d in pairs:
    ax.scatter(d["dry_score"], d["log2"], s=42, color=COLORS[d["group6"]], edgecolor="black", linewidth=0.45, alpha=0.88, zorder=3)

coef = np.polyfit(x_pair, y_pair, 1)
xx = np.linspace(min(x_pair) - 0.02, max(x_pair) + 0.02, 100)
ax.plot(xx, coef[0] * xx + coef[1], color="black", linewidth=1.5, linestyle="--")

ax.set_xlabel("circExor EV-prediction score")
ax.set_ylabel("qPCR log$_2$(EVs/Cellular), all valid pairs")
# ax.set_title("Exploratory pooled pair-level correlation", fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

legend_labels = {
    "Dry EV-high": "Predicted EV targets",
    "Dry EV-low": "Predicted Cellular targets"
}

ax.legend(
    handles=[Patch(facecolor=COLORS[k], edgecolor="black", label=legend_labels[k]) for k in ["Dry EV-high", "Dry EV-low"]],
    frameon=False, loc="upper left",
)
ax.text(
    0.98, 0.04,
    f"Pearson r={pearson_pair.statistic:.3f}, one-sided p={pearson_pair.pvalue:.3g}\nSpearman rho={spearman_pair.statistic:.3f}, one-sided p={spearman_pair.pvalue:.3g}",
    transform=ax.transAxes, ha="right", va="bottom", fontsize=9,
)

save_and_show(fig, "Fig6_dry_score_vs_qPCR_all_valid_pairs_correlation_warm_cool")


In [ ]:
# Figure 7: Zirak Figure 2E-style validation map.
#
# This adapts the logic of Zirak et al. Figure 2E to the current qPCR dataset.
#
# x-axis:
#   EV mean Ct = mean target Ct in EVs after Ct=0 is treated as ND and excluded.
#   Lower x means the target is more readily detected in EVs.
#
# y-axis:
#   Normalized qPCR EV enrichment = target-level mean log$_2$(EVs/Cellular), min-max scaled to 0-1
#   across targets with valid paired qPCR enrichment.
#
# Dotted thresholds:
#   The EV-low/cell-enriched targets are used as the RIX-like background.
#   Vertical line = mean(EV-low EV Ct) - 1 SD, so points to the left are better EV-detected.
#   Horizontal line = mean(EV-low enrichment) + 1 SD, then min-max normalized,
#   so points above it are more EV-enriched than the EV-low background.
#
# Points in the upper-left quadrant satisfy both constraints. Points without valid EV Ct or paired
# enrichment are not forced to zero; they are listed as ND/excluded in the console output.

from collections import defaultdict

ev_ct_values_by_target = defaultdict(list)
qc_by_target = defaultdict(list)
for row in cal_rows_raw:
    if row.get("sample_group") != "EVs":
        continue
    target = str(row["target"])
    ev_ct = finite_float(row.get("target_mean_ct_ignore0"))
    if ev_ct is not None:
        ev_ct_values_by_target[target].append(ev_ct)
    if row.get("QC_flags") and row.get("QC_flags") != "PASS":
        qc_by_target[target].append(row.get("QC_flags"))

zirak_rows = []
for d in targets:
    ev_cts = ev_ct_values_by_target.get(d["target"], [])
    mean_ev_ct = mean(ev_cts) if ev_cts else None
    zirak_rows.append({
        **d,
        "mean_EV_Ct": mean_ev_ct,
        "EV_Ct_n": len(ev_cts),
        "QC_notes": "; ".join(sorted(set(qc_by_target.get(d["target"], [])))),
    })

valid_zirak = [d for d in zirak_rows if d["mean_EV_Ct"] is not None and d["mean_log2"] is not None]
y_raw = [d["mean_log2"] for d in valid_zirak]
y_min, y_max = min(y_raw), max(y_raw)
for d in zirak_rows:
    if d["mean_log2"] is None:
        d["normalized_EV_enrichment"] = None
    else:
        d["normalized_EV_enrichment"] = (d["mean_log2"] - y_min) / (y_max - y_min)

low_background = [d for d in valid_zirak if d["group6"] == "Dry EV-low"]
low_ct = [d["mean_EV_Ct"] for d in low_background]
low_log2 = [d["mean_log2"] for d in low_background]
ct_threshold = mean(low_ct) - sd(low_ct)
raw_enrichment_threshold = mean(low_log2) + sd(low_log2)
normalized_enrichment_threshold = (raw_enrichment_threshold - y_min) / (y_max - y_min)

fig, ax = plt.subplots(figsize=(5.2, 4.8))

# for d in valid_zirak:
#     color = COLORS[d["group6"]]
#     is_pass = d["mean_EV_Ct"] <= ct_threshold and d["normalized_EV_enrichment"] >= normalized_enrichment_threshold
#     size = 95 if is_pass else 62
#     linewidth = 1.2 if is_pass else 0.55
#     ax.scatter(
#         d["mean_EV_Ct"],
#         d["normalized_EV_enrichment"],
#         s=size,
#         color=color,
#         edgecolor="black",
#         linewidth=linewidth,
#         zorder=4 if is_pass else 3,
#         alpha=0.95,
#     )
#     label_offset_y = 0.035 if d["normalized_EV_enrichment"] < 0.92 else -0.055
#     ax.text(
#         d["mean_EV_Ct"] + 0.12,
#         d["normalized_EV_enrichment"] + label_offset_y,
#         d["target"],
#         fontsize=8,
#         fontweight="bold" if is_pass else "normal",
#     )
# Manual label offsets: "target": (x_offset, y_offset).
# Add overlapping labels here to adjust their positions.
CUSTOM_OFFSETS = {
    "hsa_circ_0000788": (0.3, -0.05),  # Example offset.
    "hsa_circ_0001336": (0.3, 0.08),  # Example offset to avoid overlap.
}

for d in valid_zirak:
    color = COLORS[d["group6"]]
    is_pass = d["mean_EV_Ct"] <= ct_threshold and d["normalized_EV_enrichment"] >= normalized_enrichment_threshold
    size = 95 if is_pass else 62
    linewidth = 1.2 if is_pass else 0.55
    ax.scatter(
        d["mean_EV_Ct"],
        d["normalized_EV_enrichment"],
        s=size,
        color=color,
        edgecolor="black",
        linewidth=linewidth,
        zorder=4 if is_pass else 3,
        alpha=0.95,
    )
    
    # Calculate the default offset.
    label_offset_x = 0.12
    label_offset_y = 0.035 if d["normalized_EV_enrichment"] < 0.92 else -0.055
    
    # Apply manually configured offsets.
    if d["target"] in CUSTOM_OFFSETS:
        dx, dy = CUSTOM_OFFSETS[d["target"]]
        label_offset_x = dx
        label_offset_y = dy

    ax.text(
        d["mean_EV_Ct"] + label_offset_x,
        d["normalized_EV_enrichment"] + label_offset_y,
        d["target"],
        fontsize=8,
        fontweight="bold" if is_pass else "normal",
    )

ax.axvline(ct_threshold, color="black", linestyle=":", linewidth=1.7)
ax.axhline(normalized_enrichment_threshold, color="black", linestyle=":", linewidth=1.7)
ax.fill_betweenx(
    [normalized_enrichment_threshold, 1.05],
    min(d["mean_EV_Ct"] for d in valid_zirak) - 1,
    ct_threshold,
    color=cool_colors[0],
    alpha=0.18,
    zorder=0,
)

ax.set_xlabel("EV mean Ct")
ax.set_ylabel("Normalized qPCR EV enrichment")
# ax.set_title("Zirak-style EV detection and enrichment map", fontweight="bold")
ax.set_ylim(-0.03, 1.05)
ax.set_xlim(min(d["mean_EV_Ct"] for d in valid_zirak) - 0.8, max(d["mean_EV_Ct"] for d in valid_zirak) + 0.8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(
    handles=[
        Patch(facecolor=COLORS["Dry EV-high"], edgecolor="black", label="Predicted EV targets"),
        Patch(facecolor=COLORS["Dry EV-low"], edgecolor="black", label="Predicted Cellular targets"),
    ],
    frameon=False,
    loc="upper right",
    bbox_to_anchor=(1.2, 1)  # Increase x to move the legend rightward.
)

passed = [d["target"] for d in valid_zirak if d["mean_EV_Ct"] <= ct_threshold and d["normalized_EV_enrichment"] >= normalized_enrichment_threshold]
excluded = [d["target"] for d in zirak_rows if d["mean_EV_Ct"] is None or d["mean_log2"] is None]
ax.text(
    0.02,
    0.03,
    f"Ct threshold={ct_threshold:.2f}\nEnrichment threshold={normalized_enrichment_threshold:.2f}",
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=8.5,
)

print("Upper-left candidates:", ", ".join(passed) if passed else "none")
print("ND/excluded targets:", ", ".join(excluded) if excluded else "none")
for d in zirak_rows:
    print(
        d["target"],
        d["group6"],
        "EV_Ct=", None if d["mean_EV_Ct"] is None else round(d["mean_EV_Ct"], 3),
        "mean_log2=", None if d["mean_log2"] is None else round(d["mean_log2"], 3),
        "normalized=", None if d["normalized_EV_enrichment"] is None else round(d["normalized_EV_enrichment"], 3),
    )

save_and_show(fig, "Fig7_zirak_style_EV_Ct_vs_normalized_enrichment_warm_cool")
